In [1]:

import numpy as np
import pandas as pd
from scipy.special import logsumexp

modes = ["Walk", "Cycle", "PT", "Car"]

V = np.array([-2.0, -3.0, -1.0, -2.5])

nests = {
    "Active": [0, 1],
    "Motorized": [2, 3]
}


In [2]:

def nested_logit_probabilities(V, nests, lambdas):
    probabilities = np.zeros(len(V))
    nest_utilities = {}

    # Inclusive values for each nest
    for nest_name, indices in nests.items():
        lam = lambdas[nest_name]

        inclusive_value = logsumexp(V[indices] / lam)
        nest_utilities[nest_name] = lam * inclusive_value

    # Probability of choosing each nest
    names = list(nests)
    nest_values = np.array([
        nest_utilities[name] for name in names
    ])

    nest_probabilities = np.exp(
        nest_values - logsumexp(nest_values)
    )

    # Conditional probabilities within each nest
    for k, name in enumerate(names):
        indices = nests[name]
        lam = lambdas[name]

        conditional = np.exp(
            V[indices] / lam
            - logsumexp(V[indices] / lam)
        )

        probabilities[indices] = (
            nest_probabilities[k] * conditional
        )

    return probabilities


lambdas = {
    "Active": 0.7,
    "Motorized": 0.7
}

P = nested_logit_probabilities(
    V, nests, lambdas
)

display(pd.DataFrame({
    "Mode": modes,
    "Probability": P,
    "Probability (%)": P * 100
}).round(4))

print("Sum:", P.sum())


,Mode,Probability,Probability (%)
0,Walk,0.2287,22.8676
1,Cycle,0.0548,5.4802
2,PT,0.6413,64.1286
3,Car,0.0752,7.5235


Sum: 1.0000000000000002


In [3]:

from scipy.special import softmax

P_mnl = softmax(V)

P_nl = nested_logit_probabilities(
    V,
    nests,
    {"Active": 1.0, "Motorized": 1.0}
)

comparison = pd.DataFrame({
    "Mode": modes,
    "MNL": P_mnl,
    "NL (lambda=1)": P_nl,
    "Difference": P_nl - P_mnl
})

display(comparison.round(8))

assert np.allclose(P_mnl, P_nl)
print("MNL equivalence test passed.")


,Mode,MNL,NL (lambda=1),Difference
0,Walk,0.213097,0.213097,0.0
1,Cycle,0.078394,0.078394,0.0
2,PT,0.579259,0.579259,0.0
3,Car,0.129250,0.129250,0.0


MNL equivalence test passed.


In [4]:

from pathlib import Path
from scipy.optimize import minimize
from scipy.special import logsumexp, softmax
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, log_loss

PROJECT_DIR = Path.cwd().parent
df = pd.read_csv(
    PROJECT_DIR / "data" / "processed" / "lpmc.csv"
)

T = np.column_stack([
    df["dur_walking"],
    df["dur_cycling"],
    df["dur_pt_access"] + df["dur_pt_rail"]
    + df["dur_pt_bus"] + df["dur_pt_int"],
    df["dur_driving"]
])

C = np.column_stack([
    np.zeros(len(df)),
    np.zeros(len(df)),
    df["cost_transit"],
    df["cost_driving_fuel"]
    + df["cost_driving_ccharge"]
])

y = df["travel_mode"].to_numpy(dtype=int) - 1

licence = df["driving_license"].to_numpy()
ownership_1 = (df["car_ownership"] == 1).to_numpy()
ownership_2 = (df["car_ownership"] == 2).to_numpy()

person_id = (
    df["household_id"].astype(str)
    + "_"
    + df["person_n"].astype(str)
)

splitter = GroupShuffleSplit(
    n_splits=1, test_size=0.20, random_state=42
)

train_idx, test_idx = next(
    splitter.split(T, y, groups=person_id)
)

print("Training trips:", len(train_idx))
print("Test trips:", len(test_idx))


Training trips: 64982
Test trips: 16104


In [5]:

def nested_logit_predict(params, idx):

    asc_cycle, asc_pt, asc_car, b_time, b_cost, \
    b_licence, b_o1, b_o2, lam_active, lam_motor = params

    ASC = np.array([0.0, asc_cycle, asc_pt, asc_car])

    V = ASC + b_time * T[idx] + b_cost * C[idx]

    V[:, 3] += (
        b_licence * licence[idx]
        + b_o1 * ownership_1[idx]
        + b_o2 * ownership_2[idx]
    )

    P = np.zeros_like(V)

    nests = [
        ([0, 1], lam_active),
        ([2, 3], lam_motor)
    ]

    nest_log_values = []
    conditional_probabilities = []

    for alternatives, lam in nests:
        scaled_V = V[:, alternatives] / lam

        log_sum = logsumexp(
            scaled_V, axis=1
        )

        nest_log_values.append(lam * log_sum)

        conditional_probabilities.append(
            softmax(scaled_V, axis=1)
        )

    nest_log_values = np.column_stack(nest_log_values)

    nest_probabilities = softmax(
        nest_log_values, axis=1
    )

    for k, (alternatives, lam) in enumerate(nests):
        P[:, alternatives] = (
            conditional_probabilities[k]
            * nest_probabilities[:, [k]]
        )

    return P


def nested_logit_nll(params, idx):
    P = nested_logit_predict(params, idx)

    chosen_prob = P[
        np.arange(len(idx)), y[idx]
    ]

    return -np.sum(
        np.log(np.clip(chosen_prob, 1e-300, 1.0))
    )


In [6]:

parameter_names = [
    "ASC_Cycle", "ASC_PT", "ASC_Car",
    "B_Time", "B_Cost",
    "B_Licence", "B_Ownership_1", "B_Ownership_2",
    "Lambda_Active", "Lambda_Motorized"
]

initial_params = np.array([
    -3.97505, -0.59424, -4.14107,
    -5.66645, -0.18215,
    0.70825, 2.69020, 3.36303,
    0.9, 0.9
])

bounds = [(None, None)] * 8 + [
    (0.05, 1.0),
    (0.05, 1.0)
]

nl_model = minimize(
    nested_logit_nll,
    x0=initial_params,
    args=(train_idx,),
    method="L-BFGS-B",
    bounds=bounds,
    options={"maxiter": 1000}
)

print("Converged:", nl_model.success)
print("Training Log-Likelihood:", -nl_model.fun)

display(pd.DataFrame({
    "Parameter": parameter_names,
    "Estimate": nl_model.x
}).round(5))


Converged: True
Training Log-Likelihood: -51041.516900212264


,Parameter,Estimate
0,ASC_Cycle,-3.97415
1,ASC_PT,-0.59366
2,ASC_Car,-4.13872
3,B_Time,-5.66309
4,B_Cost,-0.18214
5,B_Licence,0.70804
6,B_Ownership_1,2.68890
7,B_Ownership_2,3.36186
8,Lambda_Active,1.00000
9,Lambda_Motorized,1.00000


In [7]:

# Extended MNL estimates from the previous notebook
mnl_params = np.array([
    -3.97505, -0.59424, -4.14107,
    -5.66645, -0.18215,
    0.70825, 2.69020, 3.36303
])

# Nested Logit with lambda = 1
nl_at_mnl = np.concatenate([
    mnl_params,
    [1.0, 1.0]
])

ll_at_mnl = -nested_logit_nll(
    nl_at_mnl, train_idx
)

print("Nested Logit LL at MNL parameters:", ll_at_mnl)
print("Optimized Nested Logit LL:", -nl_model.fun)
print("Difference:", -nl_model.fun - ll_at_mnl)


Nested Logit LL at MNL parameters: -51041.51502152208
Optimized Nested Logit LL: -51041.516900212264
Difference: -0.0018786901855492033


In [8]:

# Starting point: Extended MNL parameters and lambda = 1
start_params = np.concatenate([
    mnl_params,
    [1.0, 1.0]
])

# Check initial objective
initial_nll = nested_logit_nll(
    start_params, train_idx
)

# Refine optimization
nl_refined = minimize(
    nested_logit_nll,
    x0=start_params,
    args=(train_idx,),
    method="L-BFGS-B",
    bounds=bounds,
    options={
        "maxiter": 2000,
        "ftol": 1e-12,
        "gtol": 1e-6,
        "maxls": 50
    }
)

print("Converged:", nl_refined.success)
print("Initial LL:", -initial_nll)
print("Refined LL:", -nl_refined.fun)

display(pd.DataFrame({
    "Parameter": parameter_names,
    "Estimate": nl_refined.x
}).round(6))


Converged: True
Initial LL: -51041.51502152208
Refined LL: -51041.514940630856


,Parameter,Estimate
0,ASC_Cycle,-3.975062
1,ASC_PT,-0.594169
2,ASC_Car,-4.140972
3,B_Time,-5.666048
4,B_Cost,-0.182154
5,B_Licence,0.708313
6,B_Ownership_1,2.690093
7,B_Ownership_2,3.363077
8,Lambda_Active,1.000000
9,Lambda_Motorized,1.000000


In [9]:

def nested_logit_noncar_predict(params, idx):

    (
        asc_cycle, asc_pt, asc_car,
        b_time, b_cost,
        b_licence, b_o1, b_o2,
        lam_noncar
    ) = params

    ASC = np.array([0.0, asc_cycle, asc_pt, asc_car])

    V = ASC + b_time * T[idx] + b_cost * C[idx]

    V[:, 3] += (
        b_licence * licence[idx]
        + b_o1 * ownership_1[idx]
        + b_o2 * ownership_2[idx]
    )

    # Non-car nest: Walk, Cycle, PT
    logsum_noncar = logsumexp(
        V[:, :3] / lam_noncar,
        axis=1
    )

    inclusive_noncar = lam_noncar * logsum_noncar

    # Upper-level choice: Non-car versus Car
    upper_utilities = np.column_stack([
        inclusive_noncar,
        V[:, 3]
    ])

    upper_probs = softmax(upper_utilities, axis=1)

    # Conditional probabilities inside Non-car nest
    conditional_noncar = softmax(
        V[:, :3] / lam_noncar,
        axis=1
    )

    P = np.zeros_like(V)

    P[:, :3] = (
        upper_probs[:, [0]] * conditional_noncar
    )

    P[:, 3] = upper_probs[:, 1]

    return P


def nested_logit_noncar_nll(params, idx):
    P = nested_logit_noncar_predict(params, idx)

    chosen_prob = P[
        np.arange(len(idx)), y[idx]
    ]

    return -np.log(
        np.clip(chosen_prob, 1e-300, 1.0)
    ).sum()


In [10]:

# Initial parameters: Extended MNL + lambda
initial_noncar = np.concatenate([
    mnl_params,
    [0.9]
])

noncar_model = minimize(
    nested_logit_noncar_nll,
    x0=initial_noncar,
    args=(train_idx,),
    method="L-BFGS-B",
    bounds=[(None, None)] * 8 + [(0.05, 1.0)],
    options={
        "maxiter": 2000,
        "ftol": 1e-12,
        "gtol": 1e-6
    }
)

names_noncar = [
    "ASC_Cycle", "ASC_PT", "ASC_Car",
    "B_Time", "B_Cost",
    "B_Licence", "B_Ownership_1", "B_Ownership_2",
    "Lambda_NonCar"
]

print("Converged:", noncar_model.success)
print("Training Log-Likelihood:", -noncar_model.fun)

display(pd.DataFrame({
    "Parameter": names_noncar,
    "Estimate": noncar_model.x
}).round(5))


Converged: True
Training Log-Likelihood: -49900.38737895468


,Parameter,Estimate
0,ASC_Cycle,-2.47359
1,ASC_PT,-0.47432
2,ASC_Car,-3.77672
3,B_Time,-4.00671
4,B_Cost,-0.15409
5,B_Licence,0.66840
6,B_Ownership_1,2.52077
7,B_Ownership_2,3.16325
8,Lambda_NonCar,0.47421


In [11]:

from sklearn.metrics import accuracy_score, log_loss

# Predict probabilities for unseen test trips
P_nl_test = nested_logit_noncar_predict(
    noncar_model.x,
    test_idx
)

# Observed choices
y_test = y[test_idx]

# Predicted choices (highest probability)
y_pred_nl = P_nl_test.argmax(axis=1)

# Evaluation metrics
nl_accuracy = accuracy_score(y_test, y_pred_nl)

nl_logloss = log_loss(
    y_test,
    P_nl_test,
    labels=[0, 1, 2, 3]
)

print(f"Nested Logit Test Accuracy: {nl_accuracy:.4f}")
print(f"Nested Logit Test Log Loss: {nl_logloss:.4f}")

# Compare observed and predicted mode shares
mode_names = ["Walk", "Cycle", "Public Transport", "Car"]

share_comparison = pd.DataFrame({
    "Mode": mode_names,
    "Observed (%)": (
        np.bincount(y_test, minlength=4)
        / len(y_test) * 100
    ),
    "Predicted (%)": P_nl_test.mean(axis=0) * 100
})

share_comparison["Difference (pp)"] = (
    share_comparison["Predicted (%)"]
    - share_comparison["Observed (%)"]
)

display(share_comparison.round(3))


Nested Logit Test Accuracy: 0.6972
Nested Logit Test Log Loss: 0.7917


,Mode,Observed (%),Predicted (%),Difference (pp)
0,Walk,17.449,18.112,0.663
1,Cycle,3.086,2.919,-0.168
2,Public Transport,36.165,35.409,-0.756
3,Car,43.300,43.561,0.261


In [12]:

from scipy.optimize._numdiff import approx_derivative
from scipy.stats import norm

# Numerical gradient of negative log-likelihood
def nl_gradient(params):
    return approx_derivative(
        nested_logit_noncar_nll,
        params,
        args=(train_idx,),
        method="3-point"
    ).ravel()

# Numerical Hessian
H = approx_derivative(
    nl_gradient,
    noncar_model.x,
    method="3-point"
)

# Enforce numerical symmetry
H = (H + H.T) / 2

# Conventional covariance matrix
cov = np.linalg.inv(H)

# Standard errors
se = np.sqrt(np.diag(cov))

# Lambda estimate and SE
lambda_hat = noncar_model.x[-1]
lambda_se = se[-1]

# Approximate 95% Wald confidence interval
ci_lower = lambda_hat - 1.96 * lambda_se
ci_upper = lambda_hat + 1.96 * lambda_se

print(f"Lambda estimate: {lambda_hat:.5f}")
print(f"Standard Error: {lambda_se:.5f}")
print(f"95% CI: [{ci_lower:.5f}, {ci_upper:.5f}]")
print(f"Distance from 1 (in SE units): {(1-lambda_hat)/lambda_se:.2f}")


Lambda estimate: 0.47421
Standard Error: 0.00780
95% CI: [0.45893, 0.48949]
Distance from 1 (in SE units): 67.44


In [14]:

# Cluster-robust standard errors for Nested Logit

from scipy.optimize._numdiff import approx_derivative
from scipy.stats import norm

# Log-likelihood contribution of each training trip
def individual_log_likelihood(params):

    P = nested_logit_noncar_predict(
        params, train_idx
    )

    chosen_prob = P[
        np.arange(len(train_idx)),
        y[train_idx]
    ]

    return np.log(
        np.clip(chosen_prob, 1e-300, 1.0)
    )


# Numerical score matrix:
# observations x parameters
scores = approx_derivative(
    individual_log_likelihood,
    noncar_model.x,
    method="3-point"
)

# Group observations by person
train_persons = person_id.iloc[
    train_idx
].to_numpy()

score_df = pd.DataFrame(
    scores,
    columns=names_noncar
)

score_df["person_id"] = train_persons

cluster_scores = (
    score_df.groupby("person_id")[names_noncar]
    .sum()
    .to_numpy()
)

# Sandwich covariance
H_inv = np.linalg.inv(H)

meat = cluster_scores.T @ cluster_scores

cluster_cov = H_inv @ meat @ H_inv

# Finite-sample adjustment
N = len(train_idx)
G = len(cluster_scores)
K = len(noncar_model.x)

cluster_cov *= (
    G / (G - 1)
    * (N - 1) / (N - K)
)

cluster_se = np.sqrt(
    np.diag(cluster_cov)
)

# Lambda inference
lambda_hat = noncar_model.x[-1]
lambda_cluster_se = cluster_se[-1]

ci_lower = lambda_hat - 1.96 * lambda_cluster_se
ci_upper = lambda_hat + 1.96 * lambda_cluster_se

print(f"Number of clusters: {G:,}")
print(f"Lambda estimate: {lambda_hat:.5f}")
print(f"Conventional SE: {se[-1]:.5f}")
print(f"Cluster SE: {lambda_cluster_se:.5f}")

print(
    f"Cluster-robust 95% CI: "
    f"[{ci_lower:.5f}, {ci_upper:.5f}]"
)

print(
    f"Distance from lambda=1: "
    f"{(1-lambda_hat)/lambda_cluster_se:.2f} "
    f"cluster SEs"
)


Number of clusters: 25,563
Lambda estimate: 0.47421
Conventional SE: 0.00780
Cluster SE: 0.01010
Cluster-robust 95% CI: [0.45440, 0.49401]
Distance from lambda=1: 52.04 cluster SEs


In [15]:

# Nested Logit policy sensitivity analysis
# Uses the previously estimated noncar_model

mode_names = [
    "Walk", "Cycle", "Public Transport", "Car"
]

# Baseline probabilities
P_nl_base = nested_logit_noncar_predict(
    noncar_model.x, test_idx
)

# Keep original travel times and costs
T_original = T.copy()
C_original = C.copy()

try:
    # Scenario A: PT fare +10%
    C = C_original.copy()
    C[:, 2] *= 1.10

    P_nl_fare = nested_logit_noncar_predict(
        noncar_model.x, test_idx
    )

    # Scenario B: PT travel time -10%
    C = C_original.copy()
    T = T_original.copy()
    T[:, 2] *= 0.90

    P_nl_time = nested_logit_noncar_predict(
        noncar_model.x, test_idx
    )

finally:
    # Restore original variables
    T = T_original
    C = C_original


def policy_share_table(P_base, P_scenario, name):
    baseline = P_base.mean(axis=0) * 100
    scenario = P_scenario.mean(axis=0) * 100

    return pd.DataFrame({
        "Mode": mode_names,
        "Baseline (%)": baseline,
        name: scenario,
        "Change (pp)": scenario - baseline
    })


print("Nested Logit: PT Fare +10%")
display(
    policy_share_table(
        P_nl_base,
        P_nl_fare,
        "PT Fare +10% (%)"
    ).round(4)
)

print("Nested Logit: PT Time -10%")
display(
    policy_share_table(
        P_nl_base,
        P_nl_time,
        "PT Time -10% (%)"
    ).round(4)
)


Nested Logit: PT Fare +10%


,Mode,Baseline (%),PT Fare +10% (%),Change (pp)
0,Walk,18.1119,18.2966,0.1847
1,Cycle,2.9187,3.0191,0.1004
2,Public Transport,35.4086,34.8863,-0.5223
3,Car,43.5608,43.7980,0.2372


Nested Logit: PT Time -10%


,Mode,Baseline (%),PT Time -10% (%),Change (pp)
0,Walk,18.1119,16.8694,-1.2425
1,Cycle,2.9187,2.1869,-0.7318
2,Public Transport,35.4086,39.4915,4.0829
3,Car,43.5608,41.4522,-2.1086


## Summary of Findings

This analysis estimated and evaluated Nested Logit (NL) models for travel mode choice using the London Passenger Mode Choice (LPMC) dataset.

Two nesting structures were examined:

- **Active/Motorized:** Walk and Cycle versus Public Transport and Car.
- **Non-Car/Car:** Walk, Cycle, and Public Transport grouped together, with Car represented as a singleton alternative.

The Active/Motorized specification converged to dissimilarity parameters of 1, effectively reproducing the Extended MNL model.

The Non-Car/Car specification provided a substantial improvement in model fit, with a training log-likelihood of **−49,900.39**, compared with **−51,041.52** for the Extended MNL.

The estimated Non-Car dissimilarity parameter was **0.4742**, with a cluster-robust standard error of **0.0101** and an approximate 95% confidence interval of **[0.4544, 0.4940]**.

### Predictive Performance

On the held-out test dataset, Nested Logit achieved:

- **Accuracy:** 69.72%
- **Log Loss:** 0.7917

Compared with Extended MNL, the Nested Logit model achieved a lower test log loss, although classification accuracy was slightly lower.

### Behavioural Sensitivity

Two policy scenarios were evaluated:

| Scenario | Extended MNL PT Share Change | Nested Logit PT Share Change |
|---|---:|---:|
| PT fare +10% | −0.4107 pp | −0.5223 pp |
| PT travel time −10% | +3.8873 pp | +4.0829 pp |

The Nested Logit specification also predicted stronger substitution between Public Transport and the other Non-Car alternatives, Walking and Cycling.

### Conclusion

The results demonstrate that the specification of the choice correlation structure can influence both probabilistic predictive performance and estimated policy responses.

The Non-Car/Car Nested Logit specification performed better than Extended MNL in terms of held-out log loss, providing evidence that accounting for within-nest dependence may improve travel mode choice modelling.

The nesting structure should nevertheless be interpreted as a statistical representation of choice dependence rather than direct evidence of a particular sequence of travellers' decisions. Policy predictions remain model-based and require further behavioural validation.